# Spotify Songs 2023 - Exploratory Data Analysis

## Research Question
Which audio features are most strongly associated with Spotify streaming success in 2023?

## Step 1 — Research Question and Model Type (Tree-Based Approach)

### Research Question

The goal of this project is to understand and predict a song’s streaming success (`streams`) using multiple features, including audio characteristics (e.g., danceability, energy, valence), platform exposure (e.g., playlist and chart presence), and release information (e.g., year).

Instead of assuming a linear relationship between these variables and streaming success, we aim to explore whether the outcome is driven by nonlinear patterns and threshold effects.

---

### Model Type

This is a **regression problem** because the target variable `streams` is continuous (a numeric quantity).

To model this relationship, we use **tree-based models**, specifically:

* Decision Tree Regressor
* Random Forest Regressor

These models do not assume linear relationships and can capture more complex patterns in the data.

---

### Why Tree-Based Models

Tree-based models are particularly suitable for this dataset because streaming success is likely influenced by:

* **nonlinear relationships**, where changes in features do not lead to proportional changes in streams
* **threshold effects**, where passing certain exposure levels (e.g., playlist inclusion) significantly increases streams
* **interactions between variables**, where combinations of features jointly affect the outcome

By using tree-based models, we can better capture these complex relationships compared to linear approaches.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

df = pd.read_csv("spotify-2023.csv", encoding="latin1")
df.head()

,track_name,artist(s)_name,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,streams,in_apple_playlists,in_apple_charts,in_deezer_playlists,in_deezer_charts,in_shazam_charts,bpm,key,mode,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%
0,Seven (feat. Latto) (Explicit Ver.),"Latto, Jung Kook",2,2023,7,14,553,147,141381703,43,263,45,10,826,125,B,Major,80,89,83,31,0,8,4
1,LALA,Myke Towers,1,2023,3,23,1474,48,133716286,48,126,58,14,382,92,C#,Major,71,61,74,7,0,10,4
2,vampire,Olivia Rodrigo,1,2023,6,30,1397,113,140003974,94,207,91,14,949,138,F,Major,51,32,53,17,0,31,6
3,Cruel Summer,Taylor Swift,1,2019,8,23,7858,100,800840817,116,207,125,12,548,170,A,Major,55,58,72,11,0,11,15
4,WHERE SHE GOES,Bad Bunny,1,2023,5,18,3133,50,303236322,84,133,87,15,425,144,A,Minor,65,23,80,14,63,11,6


## Step 2 — Data Cleaning and Preparation

### Data Cleaning

Before building a tree-based model, we need to ensure that the target variable `streams` is stored in a numeric format.

In this dataset, the `streams` column may be interpreted as a text (object) type because some values contain commas (e.g., "1,000,000"). Since machine learning models require numeric inputs, we remove these commas and convert the column into a numeric type.

We also handle missing values in the target variable by removing rows with missing `streams`, since the model cannot learn from observations without a valid target.

---

### Handling Missing Values

Dropping missing values in the target variable is a standard approach in regression problems. Unlike input features, the target variable cannot be meaningfully imputed, because doing so would introduce artificial labels and distort the learning process.

However, it is important to consider the proportion of missing data. If a large number of observations were removed, this could reduce the sample size and affect model performance. In such cases, alternative strategies (e.g., revisiting data collection or redefining the modeling task) may be necessary.

In this dataset, the number of missing values in `streams` is small, so dropping these rows is an appropriate and reliable choice.

---

### Why this step matters

Tree-based models do not require standardization or normalization, but they do require clean and valid numeric data.

Therefore, in this step, our goal is to:

* convert the target variable into a numeric format
* remove invalid or missing target values

This ensures that the model can be trained correctly in the next steps.



In [3]:
# Convert streams to numeric
df["streams"] = df["streams"].astype(str).str.replace(",", "", regex=False)
df["streams"] = pd.to_numeric(df["streams"], errors="coerce")

# Remove rows with missing target values
df = df.dropna(subset=["streams"])

# Check result
df["streams"].info()
df["streams"].head()

<class 'pandas.Series'>
Index: 952 entries, 0 to 952
Series name: streams
Non-Null Count  Dtype  
--------------  -----  
952 non-null    float64
dtypes: float64(1)
memory usage: 14.9 KB


0    141381703.0
1    133716286.0
2    140003974.0
3    800840817.0
4    303236322.0
Name: streams, dtype: float64

## Step 3 — Feature Selection

### Feature Selection

In this step, we define the input variables (features) that will be used to predict the target variable `streams`.

Rather than relying on a single type of variable, we group the features into three categories:

* **Audio Features**: characteristics of the song itself (e.g., danceability, energy, valence)
* **Exposure Features**: measures of platform visibility (e.g., playlist inclusion and chart presence)
* **Time Features**: information about when the song was released

---

### Why this step matters

Feature selection is especially important for tree-based models because the model determines how to split the data based on these variables.

By including different types of features, the model can:

* capture **nonlinear relationships** between features and streaming success
* identify **threshold effects**, where certain variables (e.g., playlist counts) lead to large changes in streams
* learn **interactions between variables**, such as how exposure and audio features jointly influence outcomes

In particular, exposure features are expected to play a major role, as streaming success is often driven by visibility on platforms rather than only musical characteristics.

---

### Final Feature Set

The final model will use a combination of audio, exposure, and time-related variables to predict `streams`.


In [4]:
# Audio features
audio_features = [
    "danceability_%", "energy_%", "valence_%",
    "acousticness_%", "instrumentalness_%",
    "liveness_%", "speechiness_%"
]

# Exposure features
exposure_features = [
    "in_spotify_playlists", "in_spotify_charts",
    "in_apple_playlists", "in_deezer_playlists",
    "in_shazam_charts"
]

# Time features
time_features = ["released_year"]

# Combine features
features = audio_features + exposure_features + time_features

# Define X and y
X = df[features]
y = df["streams"]

# Check shape
X.shape, y.shape

((952, 13), (952,))

## Step 4 — Train/Test Split

### Train/Test Split

In this step, we split the dataset into a training set and a testing set.

* The **training set** is used to train the model
* The **testing set** is used to evaluate how well the model performs on unseen data

---

### Why this step matters

Splitting the data is essential for evaluating model performance in a realistic way.

If we train and test the model on the same data, the results may appear artificially good. This is especially important for tree-based models, which can easily overfit the training data.

By using a separate test set, we can better assess how well the model generalizes to new data.

---

### Split Ratio

We use an 80/20 split:

* 80% for training
* 20% for testing

This is a common choice that balances training size and evaluation reliability.


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Check shapes
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((761, 13), (191, 13), (761,), (191,))

## Step 5 — Feature Cleaning, Missing Value Imputation, and Tuned Decision Tree Model

### Feature Cleaning

Before training the model, all selected features must be stored in a valid numeric format.

Some variables may be read as text because of formatting issues such as commas or non-numeric entries. Since tree-based models require numeric inputs, we first convert all selected features into numeric values. Invalid entries are coerced into missing values.

---

### Handling Missing Values in Features

After converting the features to numeric format, we handle missing values using **median imputation**.

For each feature:

* the median is computed using the training data only
* missing values in the training set are filled with the training median
* missing values in the test set are filled using the same training median

This prevents **data leakage** and ensures that the data preparation process is applied consistently.

Median imputation is appropriate here because several variables, such as playlist counts and chart measures, may be skewed and influenced by extreme values.

---

### Tuned Decision Tree Model

Instead of using an unrestricted decision tree, we directly train a **tuned decision tree model**.

A tuned decision tree controls model complexity by adjusting parameters such as:

* `max_depth`
* `min_samples_split`
* `min_samples_leaf`

This helps reduce over-complexity and improves the model’s ability to generalize to unseen data.

We use **GridSearchCV** to search for the best combination of parameters based on cross-validated performance.

---

### Goal

The goal of this step is to:

* clean all selected features into numeric form
* handle missing values in a principled way
* train a tuned decision tree model with better generalization performance


In [6]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV

# Selected numeric features
numeric_features = [
    "danceability_%", "energy_%", "valence_%",
    "acousticness_%", "instrumentalness_%",
    "liveness_%", "speechiness_%",
    "in_spotify_playlists", "in_spotify_charts",
    "in_apple_playlists", "in_deezer_playlists",
    "in_shazam_charts", "released_year"
]

# Create clean copies
X_train_clean = X_train.copy()
X_test_clean = X_test.copy()

# Convert all selected features to numeric
for col in numeric_features:
    X_train_clean[col] = pd.to_numeric(
        X_train_clean[col].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )
    X_test_clean[col] = pd.to_numeric(
        X_test_clean[col].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

# Fill missing values using training medians
X_train_filled = X_train_clean.copy()
X_test_filled = X_test_clean.copy()

medians = X_train_filled.median()
X_train_filled = X_train_filled.fillna(medians)
X_test_filled = X_test_filled.fillna(medians)

# Define parameter grid
param_grid = {
    "max_depth": [3, 5, 7, 10],
    "min_samples_split": [2, 10, 20],
    "min_samples_leaf": [1, 5, 10]
}

# Grid search for tuned decision tree
grid = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

grid.fit(X_train_filled, y_train)

# Best tuned model
best_tree = grid.best_estimator_
best_tree

,"criterion criterion: {""squared_error"", ""friedman_mse"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in the half mean Poisson deviance to find splits... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_le

## Step 6 — Evaluation of the Tuned Decision Tree Model

### Model Evaluation

In this step, we evaluate the performance of the tuned decision tree model on the test set.

To measure predictive performance, we use three standard regression metrics:

* **R² (coefficient of determination)**: measures how much of the variation in `streams` is explained by the model
* **MAE (Mean Absolute Error)**: measures the average absolute prediction error
* **RMSE (Root Mean Squared Error)**: measures prediction error while giving more weight to large errors

---

### Why this step matters

The purpose of this step is to determine how well the tuned model performs on unseen data.

Because the decision tree has already been optimized using cross-validation, this evaluation provides a more meaningful estimate of predictive performance than an unrestricted tree.

---

### Goal

The goal of this step is to:

* evaluate the tuned decision tree on the test set
* assess how well the model predicts streaming success


In [21]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Predict on the test set
pred = best_tree.predict(X_test_filled)

# Evaluate performance
r2 = r2_score(y_test, pred)
mae = mean_absolute_error(y_test, pred)
rmse = mean_squared_error(y_test, pred) ** 0.5

print("Best parameters:", grid.best_params_)
print("R^2:", r2)
print("MAE:", mae)
print("RMSE:", rmse)

Best parameters: {'max_depth': 10, 'min_samples_leaf': 10, 'min_samples_split': 2}
R^2: 0.7484057474067556
MAE: 153163145.76299945
RMSE: 248170230.49849445


### Interpretation of Results

The tuned decision tree model achieves an R² of approximately 0.75, indicating that it explains around 75% of the variation in streaming counts. This suggests that the model has strong predictive power and captures the main patterns in the data.

The MAE is about 15 million streams, meaning that on average, the model’s predictions differ from the actual values by this amount. Given that many songs have streaming counts in the hundreds of millions, this level of error is reasonable.

The RMSE is higher than the MAE, which indicates that some predictions have larger errors. This is expected, as extremely popular songs are more difficult to predict accurately.

Overall, the results show that the tuned decision tree model performs well and is able to capture important factors that drive streaming success.


## Step 7 — Feature Importance of the Tuned Decision Tree

### Feature Importance

One key advantage of tree-based models is that they provide a natural way to measure **feature importance**.

Feature importance indicates how much each variable contributes to the model’s predictions. In a decision tree, features that are used more frequently and lead to larger reductions in prediction error receive higher importance scores.

---

### Why this step matters

This step allows us to interpret the model and understand which factors are most influential in determining streaming success.

By analyzing feature importance, we can identify whether:

* streaming success is driven primarily by **audio characteristics**, or
* it is more strongly influenced by **platform exposure and visibility**

---

### Goal

The goal of this step is to:

* rank the importance of all features used in the model
* identify the most influential variables
* interpret what drives streaming success based on the model


In [7]:
# Create feature importance DataFrame
importance_df = pd.DataFrame({
    "feature": numeric_features,
    "importance": best_tree.feature_importances_
})

# Sort by importance
importance_df = importance_df.sort_values("importance", ascending=False)

importance_df

,feature,importance
10,in_deezer_playlists,0.681820
7,in_spotify_playlists,0.163621
9,in_apple_playlists,0.074340
8,in_spotify_charts,0.029907
12,released_year,0.027461
3,acousticness_%,0.014331
5,liveness_%,0.003982
1,energy_%,0.002069
0,danceability_%,0.001285
11,in_shazam_charts,0.000741


### Interpretation of Feature Importance

The feature importance results show a clear pattern in how the model predicts streaming success.

The most important variable is `in_deezer_playlists`, which accounts for a very large proportion of the model’s decision-making. This indicates that playlist exposure on Deezer plays a dominant role in predicting streams.

Other exposure-related variables, such as `in_spotify_playlists` and `in_apple_playlists`, also contribute significantly to the model. Together, these variables suggest that platform visibility is a key driver of streaming success.

In contrast, audio features such as danceability, energy, and valence have very low importance scores. This indicates that the intrinsic musical characteristics of a song have relatively little influence on predicting streams in this model.

Overall, the results suggest that streaming success is driven much more by **platform exposure and distribution** than by **audio attributes alone**.


## Final Conclusion

### Summary of Findings

In this project, we used a tuned decision tree model to analyze and predict song streaming success based on audio features, platform exposure, and release information.

The model achieved strong predictive performance, with an R² of approximately 0.75, indicating that it was able to explain a substantial portion of the variation in streaming counts.

---

### Key Insights

The feature importance analysis reveals that **platform exposure variables dominate the prediction of streaming success**.

In particular, `in_deezer_playlists` is by far the most influential feature, followed by other exposure-related variables such as `in_spotify_playlists` and `in_apple_playlists`. This suggests that visibility on music platforms plays a critical role in determining how widely a song is streamed.

In contrast, audio features such as danceability, energy, and valence contribute very little to the model’s predictions. This indicates that, within this dataset, musical characteristics alone are not strong predictors of streaming success.

---

### Interpretation

These findings suggest that streaming success is driven more by **distribution and exposure mechanisms** than by intrinsic audio qualities.

Songs that receive greater playlist placement and platform visibility are much more likely to achieve high streaming counts, regardless of their specific musical attributes.

---

### Limitations

While the model performs well, there are some limitations:

* The model may struggle to accurately predict extremely popular songs, as reflected in the larger RMSE
* The dataset does not capture all possible factors influencing success, such as marketing efforts or social media trends
* The results are based on a specific dataset and may not generalize to all music markets

---

### Final Takeaway

Overall, the analysis shows that **getting exposure on streaming platforms is more important than audio characteristics in driving streaming success**.

This highlights the importance of platform algorithms, playlist curation, and distribution strategies in the modern music industry.
